In [5]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix, roc_auc_score, precision_score, classification_report
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam, Adamax
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Flatten, Dropout, Conv2D, Input, Lambda, BatchNormalization, Activation, MaxPooling2D
from tensorflow.keras.applications import Xception

class DataLoader:
    def __init__(self, base_path, dataset_type=''):
        self.base_path = base_path
        self.dataset_type = dataset_type
        self.image_extensions = ['.png', '.jpg', '.jpeg']
        self.paths = self._define_paths()

    def _define_paths(self):
        if self.dataset_type == 'chest_xray':
            dirs = ['NORMAL', 'PNEUMONIA']
        else:
            dirs = ['normal', 'adenocarcinoma', 'large.cell.carcinoma', 'squamous.cell.carcinoma']
        
        types = ['train', 'test', 'valid']
        return {t: {d: os.path.join(self.base_path, t, d) for d in dirs} for t in types}

    def get_files_with_extensions(self, directory):
        files = []
        for ext in self.image_extensions:
            files.extend(glob.glob(os.path.join(directory, '*' + ext)))
        return [x.replace('\\', '/') for x in files]

    def load_data(self):
        data = {t: {d: self.get_files_with_extensions(p) for d, p in self.paths[t].items()} for t in self.paths}
        return data

    def print_data_summary(self, data):
        for t in data:
            total = sum(len(data[t][d]) for d in data[t])
            normal_key = 'NORMAL' if 'NORMAL' in data[t] else 'normal'
            print(f'{t}:', ', '.join([f'{d}({len(data[t][d])})' for d in data[t]]),
                  f'Healthy as % of total: {100*round(len(data[t][normal_key]) / total, 2)}%')


class DataProcessor:
    def __init__(self, data):
        self.data = data

    def create_labelled_data(self, label_mapping):
        list_total = []
        for t in self.data:
            for label in self.data[t]:
                list_total.extend([[img, label_mapping[label]] for img in self.data[t][label]])
        df_total = pd.DataFrame(list_total, columns=['image', 'label']).sample(frac=1).reset_index(drop=True)
        return df_total

    @staticmethod
    def process_image(img_path):
        img = cv2.imread(img_path)
        img = cv2.resize(img, (256, 256))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img = img / 255.0
        img = np.reshape(img, (256,256,1))
        return img

    def compose_dataset(self, df):
        data = []
        labels = []
        for img_path, label in df.values:
            data.append(self.process_image(img_path))
            labels.append(label)
        return np.array(data), np.array(labels)
    
    @staticmethod
    def map_labels(labels, mapping):
        return np.vectorize(lambda x: mapping.get(x, x))(labels)


class DataSplitter:
    @staticmethod
    def split_data(X, y, test_size=0.3, val_size=0.5, random_state=2021, suffix=''):
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
        X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=val_size, random_state=random_state)
        #return X_train, X_val, X_test, y_train, y_val, y_test
    
        split = {
            f'X_train{suffix}': X_train,
            f'X_val{suffix}': X_val,
            f'X_test{suffix}': X_test,
            f'y_train{suffix}': y_train,
            f'y_val{suffix}': y_val,
            f'y_test{suffix}': y_test
            }
        return split


class DataMerger:
    def __init__(self, data_dict):
        self.data_dict = data_dict

    def concatenate_datasets(self, set_type, suffixes):
        # Konkatenacja danych obrazów
        X_combined = np.concatenate(
            [self.data_dict.get(f'X_{set_type}{suffix}') for suffix in suffixes], axis=0
        )
        # Konkatenacja etykiet
        Y_combined = np.concatenate(
            [self.data_dict.get(f'y_{set_type}{suffix}_mapped') for suffix in suffixes], axis=0
        )
        return X_combined, Y_combined

    def prepare_combined_datasets(self, suffixes):
        combined_datasets = {}
        for set_type in ['train', 'test', 'val']:
            X, Y = self.concatenate_datasets(set_type, suffixes)
            combined_datasets[f'X_{set_type}'] = X
            combined_datasets[f'Y_{set_type}'] = Y
        return combined_datasets


class ModelAnalysis:
    def __init__(self, model=None):
        self.model = model

    def train_summary(self, history):
        plt.figure(figsize=(20,5))

        # plot loss & val loss
        plt.subplot(1,2,1)
        sns.lineplot(x=history.epoch, y=history.history['loss'], color='red', label='Train Loss')
        sns.lineplot(x=history.epoch, y=history.history['val_loss'], color='orange', label='Test Loss')
        plt.title('Loss on train vs test')
        plt.legend(loc='best')

        # plot accuracy and val accuracy
        plt.subplot(1,2,2)
        sns.lineplot(x=history.epoch, y=history.history['accuracy'], color='blue', label='Train Accuracy')
        sns.lineplot(x=history.epoch, y=history.history['val_accuracy'], color='green', label='Test Accuracy')
        plt.title('Accuracy on train vs test')
        plt.legend(loc='best')

        plt.show()

    def model_summary(self, X, y):
        y_pred = self.model.model.predict(X, batch_size=4)
        y_pred_classes = np.argmax(y_pred, axis=1)
        
        # Sprawdź, czy 'y' jest już w formacie etykiet klas
        if len(y.shape) > 1 and y.shape[1] > 1:
            y = np.argmax(y, axis=1)

        print(f'accuracy: {accuracy_score(y, y_pred_classes)}')
        print(f'recall: {recall_score(y, y_pred_classes, average="weighted")}')
        print(f'precision: {precision_score(y, y_pred_classes, average="weighted")}')
        print(f'f1: {f1_score(y, y_pred_classes, average="weighted")}')

        # Tworzenie macierzy konfuzji
        cf_matrix = confusion_matrix(y, y_pred_classes)
        sns.heatmap(cf_matrix, annot=True, fmt='g', cmap='Blues')

        plt.xlabel('Predicted labels')
        plt.ylabel('True labels')
        plt.title('Confusion Matrix')
        plt.show()

    def plot_label_distribution(self, y_train, y_test, y_val, label_map, figsize):
        plt.figure(figsize=figsize)

        plt.subplot(1, 3, 1)
        y_train_mapped = [label_map[label] for label in y_train]
        ax = sns.countplot(x=y_train_mapped, hue=y_train_mapped, palette='viridis', dodge=False, legend=False)
        plt.title(f'Zbiór treningowy', fontsize=16)
        plt.xlabel('')
        plt.ylabel('')
        ax.set_xticks(range(len(ax.get_xticklabels())))
        ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=14)
        for p in ax.patches:
            ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='baseline', fontsize=12, color='black', xytext=(0, 5), textcoords='offset points')


        plt.subplot(1, 3, 2)
        y_test_mapped = [label_map[label] for label in y_test]
        ax = sns.countplot(x=y_test_mapped, hue=y_test_mapped, palette='viridis', dodge=False, legend=False)
        plt.title(f'Zbiór testowy', fontsize=16)
        plt.xlabel('')
        plt.ylabel('')
        ax.set_xticks(range(len(ax.get_xticklabels())))
        ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=14)
        for p in ax.patches:
            ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='baseline', fontsize=12, color='black', xytext=(0, 5), textcoords='offset points')

        plt.subplot(1, 3, 3)
        y_val_mapped = [label_map[label] for label in y_val]
        ax = sns.countplot(x=y_val_mapped, hue=y_val_mapped, palette='viridis', dodge=False, legend=False)
        plt.title(f'Zbiór walidacyjny', fontsize=16)
        plt.xlabel('')
        plt.ylabel('')
        ax.set_xticks(range(len(ax.get_xticklabels())))
        ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=14)
        for p in ax.patches:
            ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='baseline', fontsize=12, color='black', xytext=(0, 5), textcoords='offset points')
        plt.tight_layout()
        plt.show()


class CNN:
    def __init__(self, input_shape, num_classes, patience):
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.patience = patience
        self.model = self.build_model()
        
    def build_model(self):
        model = Sequential()
        model.add(Conv2D(32, (7, 7), padding='same', activation='relu', input_shape=self.input_shape))
        model.add(Conv2D(32, (7, 7), padding='same', activation='relu'))
        model.add(MaxPooling2D((2, 2)))
        
        model.add(Conv2D(64, (5, 5), padding='same', activation='relu'))
        model.add(Conv2D(64, (5, 5), padding='same', activation='relu'))
        model.add(MaxPooling2D((2, 2)))
        
        model.add(Conv2D(128, (3, 3), padding='same', activation='relu'))
        model.add(Conv2D(128, (3, 3), padding='same', activation='relu'))
        model.add(MaxPooling2D((2, 2)))
        
        model.add(Flatten())
        model.add(Dense(128, activation='relu'))
        model.add(Dropout(0.33))
        
        model.add(Dense(self.num_classes, activation='softmax'))
        
        return model

    def compile_and_train(self, X_train, Y_train, X_val, Y_val, epochs=100, learning_rate=0.0001):
        self.model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=learning_rate), metrics=['accuracy'])
        callback = EarlyStopping(monitor='val_loss', patience=self.patience, verbose=1)
        history = self.model.fit(X_train, Y_train, validation_data=(X_val, Y_val), epochs=epochs, verbose=1, callbacks=[callback])
        return history

    def get_model_summary(self):
        return self.model.summary()


class EfficientNetB3:
    def __init__(self, input_shape, num_classes, patience):
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.patience = patience
        self.model = self.build_model()
        
    def build_model(self):
        inputs = Input(shape=self.input_shape)
        
        # Konwertuj 1 kanał na 3 kanały
        x = Lambda(lambda x: tf.image.grayscale_to_rgb(x))(inputs)
        #x = Conv2D(3, (3, 3), padding='same', activation='relu')(inputs)
        
        base_model = tf.keras.applications.EfficientNetB3(include_top=False, 
                                                          weights='imagenet', 
                                                          input_shape=(self.input_shape[0], self.input_shape[1], 3), 
                                                          pooling='max')
        for layer in base_model.layers:
            layer.trainable = False
        
        x = base_model(x)
        x = Flatten()(x)
        x = Dense(128, activation='relu')(x)
        x = Dropout(0.33)(x)
        outputs = Dense(self.num_classes, activation='softmax')(x)
        
        model = Model(inputs, outputs)
        
        return model

    def compile_and_train(self, X_train, Y_train, X_val, Y_val, epochs=100, learning_rate=0.0001):
        self.model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=learning_rate), metrics=['accuracy'])
        callback = EarlyStopping(monitor='val_loss', patience=self.patience, verbose=1)
        history = self.model.fit(X_train, Y_train, validation_data=(X_val, Y_val), epochs=epochs, verbose=1, callbacks=[callback])
        return history
    
    def get_model_summary(self):
        return self.model.summary()

class VGG16:
    def __init__(self, input_shape, num_classes, patience):
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.patience = patience
        self.model = self.build_model()

    def build_model(self):
        model = Sequential()
        
        # Blok 1
        model.add(Conv2D(64, (3, 3), padding='same', activation='relu', input_shape=self.input_shape))
        model.add(Conv2D(64, (3, 3), padding='same', activation='relu'))
        model.add(MaxPooling2D(pool_size=(2, 2)))
        
        # Blok 2
        model.add(Conv2D(128, (3, 3), padding='same', activation='relu'))
        model.add(Conv2D(128, (3, 3), padding='same', activation='relu'))
        model.add(MaxPooling2D(pool_size=(2, 2)))
        
        # Blok 3
        model.add(Conv2D(256, (3, 3), padding='same', activation='relu'))
        model.add(Conv2D(256, (3, 3), padding='same', activation='relu'))
        model.add(Conv2D(256, (3, 3), padding='same', activation='relu'))
        model.add(MaxPooling2D(pool_size=(2, 2)))
        
        # Blok 4
        model.add(Conv2D(512, (3, 3), padding='same', activation='relu'))
        model.add(Conv2D(512, (3, 3), padding='same', activation='relu'))
        model.add(Conv2D(512, (3, 3), padding='same', activation='relu'))
        model.add(MaxPooling2D(pool_size=(2, 2)))

        # Blok 5
        model.add(Conv2D(512, (3, 3), padding='same', activation='relu'))
        model.add(Conv2D(512, (3, 3), padding='same', activation='relu'))
        model.add(Conv2D(512, (3, 3), padding='same', activation='relu'))
        model.add(MaxPooling2D(pool_size=(2, 2)))
        
        # Główne bloki
        model.add(Flatten())
        model.add(Dense(4096, activation='relu'))
        model.add(Dropout(0.5))
        model.add(Dense(4096, activation='relu'))
        model.add(Dropout(0.5))
        model.add(Dense(self.num_classes, activation='softmax'))
        
        return model

    def compile_and_train(self, X_train, Y_train, X_val, Y_val, epochs=100, learning_rate=0.0001):
        self.model.compile(optimizer=Adam(learning_rate=learning_rate), loss='categorical_crossentropy', metrics=['accuracy'])
        callback = EarlyStopping(monitor='val_loss', patience=self.patience, verbose=1)
        history = self.model.fit(X_train, Y_train, validation_data=(X_val, Y_val), epochs=epochs, verbose=1, callbacks=[callback])
        return history
    
    def get_model_summary(self):
        self.model.summary()


class XceptionGrayScaleModel:
    def __init__(self, input_shape, num_classes, patience):
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.patience = patience
        self.model = self.build_model()

    def build_model(self):
        model = Sequential()

        # Warstwa konwertująca obrazy z jednego kanału na trzy kanały
        model.add(Input(shape=self.input_shape))
        model.add(Lambda(lambda x: tf.image.grayscale_to_rgb(x)))

        # Bazowy model Xception
        base_model = Xception(weights='imagenet', include_top=False, pooling='avg', input_shape=(256, 256, 3))

        # Zamrożenie wag w bazowym modelu
        base_model.trainable = False

        # Dodanie bazowego modelu do modelu sekwencyjnego
        model.add(base_model)

        # Batch Normalization
        model.add(BatchNormalization())

        # Dropout Layer
        model.add(Dropout(0.45))

        # Dense Layer 1
        model.add(Dense(220, activation='relu'))

         # Dropout Layer
        model.add(Dropout(0.25))

        # Dense Layer 2
        model.add(Dense(60, activation='relu'))

        # Output Layer
        model.add(Dense(self.num_classes, activation='sigmoid'))

        return model
        
    def compile_and_train(self, X_train, Y_train, X_val, Y_val, epochs=100, learning_rate=0.001):
        self.model.compile(optimizer=Adamax(learning_rate=learning_rate), loss='binary_crossentropy', metrics=['accuracy'])
        callback = EarlyStopping(monitor='val_loss', patience=self.patience, verbose=1)
        history = self.model.fit(X_train, Y_train, validation_data=(X_val, Y_val), epochs=epochs, verbose=1, callbacks=[callback])
        return history
    
    def get_model_summary(self):
        self.model.summary()




def main():
    # Path for different datasets
    chest_xray_path = 'C:\\Users\\User\\Desktop\\Praca_magisterska\\chest_xray\\'
    general_data_path = 'C:\\Users\\User\\Desktop\\Praca_magisterska\\Data\\'

    # Load chest xray data
    chest_xray_loader = DataLoader(chest_xray_path, dataset_type='chest_xray')
    chest_xray_data = chest_xray_loader.load_data()
    chest_xray_loader.print_data_summary(chest_xray_data)

    # Load general data
    general_loader = DataLoader(general_data_path)
    general_data = general_loader.load_data()
    general_loader.print_data_summary(general_data)

    # Process chest xray data
    chest_xray_processor = DataProcessor(chest_xray_data)
    chest_xray_label_mapping = {
        'NORMAL': 0,
        'PNEUMONIA': 1
    }
    chest_xray_df_total = chest_xray_processor.create_labelled_data(chest_xray_label_mapping)
    X_chest_xray, y_chest_xray = chest_xray_processor.compose_dataset(chest_xray_df_total)
    print('Chest X-ray data shape: {}, Labels shape: {}'.format(X_chest_xray.shape, y_chest_xray.shape))

    # Process general data
    tomography_processor = DataProcessor(general_data)
    tomography_label_mapping = {
        'normal': 2,
        'adenocarcinoma': 3,
        'large.cell.carcinoma': 4,
        'squamous.cell.carcinoma': 5
    }
    tomography_df_total = tomography_processor.create_labelled_data(tomography_label_mapping)
    X_tomography, y_tomography = tomography_processor.compose_dataset(tomography_df_total)
    print('Tomography data shape: {}, Labels shape: {}'.format(X_tomography.shape, y_tomography.shape))
    

    rtg = DataSplitter.split_data(X_chest_xray, y_chest_xray, suffix='_rtg')
    tomografia = DataSplitter.split_data(X_tomography, y_tomography, suffix='_tomografia')


    print('Train data shape: {}, Labels shape: {}'.format(rtg.get('X_train_rtg').shape, rtg.get('y_train_rtg').shape))
    print('Test data shape: {}, Labels shape: {}'.format(rtg.get('X_test_rtg').shape, rtg.get('y_test_rtg').shape))
    print('Valid data shape: {}, Labels shape: {}'.format(rtg.get('X_val_rtg').shape, rtg.get('y_val_rtg').shape))
    print('Train data shape: {}, Labels shape: {}'.format(tomografia.get('X_train_tomografia').shape, tomografia.get('y_train_tomografia').shape))
    print('Test data shape: {}, Labels shape: {}'.format(tomografia.get('X_test_tomografia').shape, tomografia.get('y_test_tomografia').shape))
    print('Valid data shape: {}, Labels shape: {}'.format(tomografia.get('X_val_tomografia').shape, tomografia.get('y_val_tomografia').shape))

    analysis = ModelAnalysis()
    label_map = {0: 'Zdrowy', 1: 'Chory', 2: 'Zdrowy', 3:'Chory na raka gruczołowego', 4:'Chory na raka wielokomórowego', 5:'Chory na raka kolczystokomórkowego'}

    #analysis.plot_label_distribution(tomografia.get('y_train_tomografia'), tomografia.get('y_test_tomografia'), tomografia.get('y_val_tomografia'), label_map=label_map, figsize=(22, 9.5))
    #analysis.plot_label_distribution(rtg.get('y_train_rtg'), rtg.get('y_test_rtg'), rtg.get('y_val_rtg'), label_map=label_map, figsize=(18, 6.5))


    label_mapping = {
        0: 0, 1: 0, 2:1, 3:1, 4:1, 5:1
    }

    processor = DataProcessor(None)
    y_train_rtg_mapped = processor.map_labels(rtg.get('y_train_rtg'), label_mapping)
    y_test_rtg_mapped = processor.map_labels(rtg.get('y_test_rtg'), label_mapping)
    y_val_rtg_mapped = processor.map_labels(rtg.get('y_val_rtg'), label_mapping)

    y_train_tomografia_mapped = processor.map_labels(tomografia.get('y_train_tomografia'), label_mapping)
    y_test_tomografia_mapped = processor.map_labels(tomografia.get('y_test_tomografia'), label_mapping)
    y_val_tomografia_mapped = processor.map_labels(tomografia.get('y_val_tomografia'), label_mapping)

    # Przykład przekazania danych do klasy DataMerger
    all_data = {
    'X_test_tomografia': tomografia.get('X_test_tomografia'),
    'X_test_rtg': rtg.get('X_test_rtg'),
    'y_test_tomografia_mapped': y_test_tomografia_mapped,
    'y_test_rtg_mapped': y_test_rtg_mapped,
    'X_train_tomografia': tomografia.get('X_train_tomografia'),
    'X_train_rtg': rtg.get('X_train_rtg'),
    'y_train_tomografia_mapped': y_train_tomografia_mapped,
    'y_train_rtg_mapped': y_train_rtg_mapped,
    'X_val_tomografia': tomografia.get('X_val_tomografia'),
    'X_val_rtg': rtg.get('X_val_rtg'),
    'y_val_tomografia_mapped': y_val_tomografia_mapped,
    'y_val_rtg_mapped': y_val_rtg_mapped
    }

    merger = DataMerger(all_data)
    combined_data = merger.prepare_combined_datasets(['_tomografia', '_rtg'])

    # Dostęp do połączonych danych
    X_combined_test = combined_data['X_test']
    y_combined_test = combined_data['Y_test']
    X_combined_train = combined_data['X_train']
    y_combined_train = combined_data['Y_train']
    X_combined_val = combined_data['X_val']
    y_combined_val = combined_data['Y_val']


    #obrobka etykiet zbiorow polaczonych
    y_combined_train = to_categorical(y_combined_train)
    y_combined_test = to_categorical(y_combined_test)
    y_combined_val = to_categorical(y_combined_val)

    #tomografia dane
    X_train_tomografia=tomografia.get('X_train_tomografia')
    X_test_tomografia=tomografia.get('X_test_tomografia')
    X_val_tomografia=tomografia.get('X_val_tomografia')
    y_train_tomografia=to_categorical(tomografia.get('y_train_tomografia')-2)
    y_test_tomografia= to_categorical(tomografia.get('y_test_tomografia')-2)
    y_val_tomografia = to_categorical(tomografia.get('y_val_tomografia')-2)

    #rtg dane
    X_train_rtg=rtg.get('X_train_rtg')
    X_test_rtg=rtg.get('X_test_rtg')
    X_val_rtg=rtg.get('X_val_rtg')
    y_train_rtg=to_categorical(rtg.get('y_train_rtg'))
    y_test_rtg= to_categorical(rtg.get('y_test_rtg'))
    y_val_rtg = to_categorical(rtg.get('y_val_rtg'))

    #CNN
    # CNN dla  danych połączonych
    CNN_model = CNN(input_shape=(256, 256, 1), num_classes=2, patience=3)
    # Wyświetlanie podsumowania modelu
    CNN_model.get_model_summary()
    
    history_CNN_model = CNN_model.compile_and_train(X_combined_train, y_combined_train, X_combined_test, y_combined_test, epochs=100)

    # Podsumowanie
    analysis = ModelAnalysis(CNN_model)
    analysis.train_summary(history_CNN_model)
    analysis.model_summary(X_combined_val, y_combined_val)

    CNN_model = CNN(input_shape=(256, 256, 1), num_classes=2, patience=5)
    history_CNN_model = CNN_model.compile_and_train(X_combined_train, y_combined_train, X_combined_val, y_combined_val, epochs=100)

    analysis = ModelAnalysis(CNN_model)
    analysis.train_summary(history_CNN_model)
    analysis.model_summary(X_combined_test, y_combined_test)

    CNN_model = CNN(input_shape=(256, 256, 1), num_classes=2, patience=10)
    history_CNN_model = CNN_model.compile_and_train(X_combined_train, y_combined_train, X_combined_val, y_combined_val, epochs=100)

    analysis = ModelAnalysis(CNN_model)
    analysis.train_summary(history_CNN_model)
    analysis.model_summary(X_combined_test, y_combined_test)
   

    # CNN dla  tomografii
    CNN_model = CNN(input_shape=(256, 256, 1), num_classes=4, patience=3)
    history_CNN_model= CNN_model.compile_and_train(X_train_tomografia, y_train_tomografia, X_val_tomografia, y_val_tomografia, epochs=100)

    analysis = ModelAnalysis(CNN_model)
    analysis.train_summary(history_CNN_model)
    analysis.model_summary(X_test_tomografia, y_test_tomografia)

    CNN_model = CNN(input_shape=(256, 256, 1), num_classes=4, patience=5)
    history_CNN_model = CNN_model.compile_and_train(X_train_tomografia, y_train_tomografia, X_val_tomografia, y_val_tomografia, epochs=100)

    analysis = ModelAnalysis(CNN_model)
    analysis.train_summary(history_CNN_model)
    analysis.model_summary(X_test_tomografia, y_test_tomografia)

    CNN_model = CNN(input_shape=(256, 256, 1), num_classes=4, patience=10)
    history_CNN_model = CNN_model.compile_and_train(X_train_tomografia, y_train_tomografia, X_val_tomografia, y_val_tomografia, epochs=100)

    analysis = ModelAnalysis(CNN_model)
    analysis.train_summary(history_CNN_model)
    analysis.model_summary(X_test_tomografia, y_test_tomografia)

    # CNN dla  rtg
    CNN_model = CNN(input_shape=(256, 256, 1), num_classes=2, patience=3)
    history_CNN_model = CNN_model.compile_and_train(X_train_rtg, y_train_rtg, X_val_rtg, y_val_rtg, epochs=100)

    analysis = ModelAnalysis(CNN_model)
    analysis.train_summary(history_CNN_model)
    analysis.model_summary(X_test_rtg, y_test_rtg)

    CNN_model = CNN(input_shape=(256, 256, 1), num_classes=2, patience=5)
    history_CNN_model = CNN_model.compile_and_train(X_train_rtg, y_train_rtg, X_val_rtg, y_val_rtg, epochs=100)

    analysis = ModelAnalysis(CNN_model)
    analysis.train_summary(history_CNN_model)
    analysis.model_summary(X_test_rtg, y_test_rtg)

    CNN_model = CNN(input_shape=(256, 256, 1), num_classes=2, patience=10)
    history_CNN_model = CNN_model.compile_and_train(X_train_rtg, y_train_rtg, X_val_rtg, y_val_rtg, epochs=100)

    analysis = ModelAnalysis(CNN_model)
    analysis.train_summary(history_CNN_model)
    analysis.model_summary(X_test_rtg, y_test_rtg)

    # Xception dla tomografii
    xception_model = XceptionGrayScaleModel(input_shape=(256, 256, 1), num_classes=4, patience=3)
    # Wyświetlanie podsumowania modelu Xception
    xception_model.get_model_summary()

    history_xception = xception_model.compile_and_train(X_train_tomografia, y_train_tomografia, X_val_tomografia, y_val_tomografia, epochs=100)

    analysis = ModelAnalysis(xception_model)
    analysis.train_summary(history_xception)
    analysis.model_summary(X_test_tomografia, y_test_tomografia)

    xception_model = XceptionGrayScaleModel(input_shape=(256, 256, 1), num_classes=4, patience=5)
    history_xception = xception_model.compile_and_train(X_train_tomografia, y_train_tomografia, X_val_tomografia, y_val_tomografia, epochs=100)

    analysis = ModelAnalysis(xception_model)
    analysis.train_summary(history_xception)
    analysis.model_summary(X_test_tomografia, y_test_tomografia)

    xception_model = XceptionGrayScaleModel(input_shape=(256, 256, 1), num_classes=4, patience=10)
    history_xception = xception_model.compile_and_train(X_train_tomografia, y_train_tomografia, X_val_tomografia, y_val_tomografia, epochs=100)

    analysis = ModelAnalysis(xception_model)
    analysis.train_summary(history_xception)
    analysis.model_summary(X_test_tomografia, y_test_tomografia)

    #xception dla rtg
    xception_model = XceptionGrayScaleModel(input_shape=(256, 256, 1), num_classes=2, patience=3)
    history_xception = xception_model.compile_and_train(X_train_rtg, y_train_rtg, X_val_rtg, y_val_rtg, epochs=100)

    analysis = ModelAnalysis(xception_model)
    analysis.train_summary(history_xception)
    analysis.model_summary(X_test_rtg, y_test_rtg)

    xception_model = XceptionGrayScaleModel(input_shape=(256, 256, 1), num_classes=2, patience=5)
    history_xception = xception_model.compile_and_train(X_train_rtg, y_train_rtg, X_val_rtg, y_val_rtg, epochs=100)

    analysis = ModelAnalysis(xception_model)
    analysis.train_summary(history_xception)
    analysis.model_summary(X_test_rtg, y_test_rtg)

    xception_model = XceptionGrayScaleModel(input_shape=(256, 256, 1), num_classes=2, patience=10)
    history_xception = xception_model.compile_and_train(X_train_rtg, y_train_rtg, X_val_rtg, y_val_rtg, epochs=100)

    analysis = ModelAnalysis(xception_model)
    analysis.train_summary(history_xception)
    analysis.model_summary(X_test_rtg, y_test_rtgX_val_rtg, y_val_rtg)

    #EfficientNetB3 dla tomografi
    EffNetB3_model = EfficientNetB3(input_shape=(256, 256, 1), num_classes=4, patience=3)
    # Wyświetlanie podsumowania modelu
    EffNetB3_model.get_model_summary()
    history_EffNetB3_model = EffNetB3_model.compile_and_train(X_train_tomografia, y_train_tomografia, X_val_tomografia, y_val_tomografia, epochs=100)

    analysis = ModelAnalysis(EffNetB3_model)
    analysis.train_summary(history_EffNetB3_model)
    analysis.model_summary(X_test_tomografia, y_test_tomografia)

    EffNetB3_model = EfficientNetB3(input_shape=(256, 256, 1), num_classes=4, patience=5)
    history_EffNetB3_model = EffNetB3_model.compile_and_train(X_train_tomografia, y_train_tomografia, X_val_tomografia, y_val_tomografia, epochs=100)

    analysis = ModelAnalysis(EffNetB3_model)
    analysis.train_summary(history_EffNetB3_model)
    analysis.model_summary(X_test_tomografia, y_test_tomografia)

    EffNetB3_model = EfficientNetB3(input_shape=(256, 256, 1), num_classes=4, patience=10)
    history_EffNetB3_model = EffNetB3_model.compile_and_train(X_train_tomografia, y_train_tomografia, X_val_tomografia, y_val_tomografia, epochs=100)

    analysis = ModelAnalysis(EffNetB3_model)
    analysis.train_summary(history_EffNetB3_model)
    analysis.model_summary(X_test_tomografia, y_test_tomografia)

    # EfficientNetB3 dla  rtg
    EffNetB3_model = EfficientNetB3(input_shape=(256, 256, 1), num_classes=2, patience=3)
    history_EffNetB3_model = EffNetB3_model.compile_and_train(X_train_rtg, y_train_rtg, X_val_rtg, y_val_rtg, epochs=100)

    analysis = ModelAnalysis(EffNetB3_model)
    analysis.train_summary(history_EffNetB3_model)
    analysis.model_summary( X_test_rtg, y_test_rtg)

    EffNetB3_model = EfficientNetB3(input_shape=(256, 256, 1), num_classes=2, patience=5)
    history_EffNetB3_model = EffNetB3_model.compile_and_train(X_train_rtg, y_train_rtg, X_val_rtg, y_val_rtg, epochs=100)

    analysis = ModelAnalysis(EffNetB3_model)
    analysis.train_summary(history_EffNetB3_model)
    analysis.model_summary(X_test_rtg, y_test_rtg)

    EffNetB3_model = EfficientNetB3(input_shape=(256, 256, 1), num_classes=2, patience=10)
    history_EffNetB3_model = EffNetB3_model.compile_and_train(X_train_rtg, y_train_rtg, X_val_rtg, y_val_rtg, epochs=100)

    analysis = ModelAnalysis(EffNetB3_model)
    analysis.train_summary(history_EffNetB3_model)
    analysis.model_summary(X_test_rtg, y_test_rtg)

    # VGG16 dla tomografii
    VGG16_model = VGG16(input_shape=(256, 256, 1), num_classes=4, patience=3)
    VGG16_model.get_model_summary()
    history_VGG16_model = VGG16_model.compile_and_train(X_train_tomografia, y_train_tomografia, X_val_tomografia, y_val_tomografia, epochs=100)

    analysis = ModelAnalysis(VGG16_model)
    analysis.train_summary(history_VGG16_model)
    analysis.model_summary(X_test_tomografia, y_test_tomografia)

    VGG16_model = VGG16(input_shape=(256, 256, 1), num_classes=4, patience=5)
    history_VGG16_model = VGG16_model.compile_and_train(X_train_tomografia, y_train_tomografia, X_val_tomografia, y_val_tomografia, epochs=100)

    analysis = ModelAnalysis(VGG16_model)
    analysis.train_summary(history_VGG16_model)
    analysis.model_summary(X_test_tomografia, y_test_tomografia)

    VGG16_model = VGG16(input_shape=(256, 256, 1), num_classes=4, patience=10)
    history_VGG16_model = VGG16_model.compile_and_train(X_train_tomografia, y_train_tomografia, X_val_tomografia, y_val_tomografia, epochs=100)

    analysis = ModelAnalysis(VGG16_model)
    analysis.train_summary(history_VGG16_model)
    analysis.model_summary(X_test_tomografia, y_test_tomografia)

    # VGG16 dla  rtg
    VGG16_model= VGG16(input_shape=(256, 256, 1), num_classes=2, patience=3)
    history_VGG16_model = VGG16_model.compile_and_train(X_train_rtg, y_train_rtg, X_val_rtg, y_val_rtg, epochs=100)

    analysis = ModelAnalysis(VGG16_model)
    analysis.train_summary(history_VGG16_model)
    analysis.model_summary(X_test_rtg, y_test_rtg)

    VGG16_model = VGG16(input_shape=(256, 256, 1), num_classes=2, patience=5)
    history_VGG16_model=VGG16_model.compile_and_train(X_train_rtg, y_train_rtg, X_val_rtg, y_val_rtg, epochs=100)

    analysis = ModelAnalysis(VGG16_model)
    analysis.train_summary(history_VGG16_model)
    analysis.model_summary(X_test_rtg, y_test_rtg)

    VGG16_model = VGG16(input_shape=(256, 256, 1), num_classes=2, patience=10)
    history_VGG16_model = VGG16_model.compile_and_train(X_train_rtg, y_train_rtg, X_val_rtg, y_val_rtg, epochs=100)

    analysis = ModelAnalysis(VGG16_model)
    analysis.train_summary(history_VGG16_model)
    analysis.model_summary(X_test_rtg, y_test_rtg)

if __name__ == "__main__":
    main()


train: NORMAL(1341), PNEUMONIA(3875) Healthy as % of total: 26.0%
test: NORMAL(234), PNEUMONIA(390) Healthy as % of total: 38.0%
valid: NORMAL(8), PNEUMONIA(8) Healthy as % of total: 50.0%
train: normal(148), adenocarcinoma(195), large.cell.carcinoma(115), squamous.cell.carcinoma(155) Healthy as % of total: 24.0%
test: normal(54), adenocarcinoma(120), large.cell.carcinoma(51), squamous.cell.carcinoma(90) Healthy as % of total: 17.0%
valid: normal(13), adenocarcinoma(23), large.cell.carcinoma(21), squamous.cell.carcinoma(15) Healthy as % of total: 18.0%


MemoryError: Unable to allocate 2.86 GiB for an array with shape (5856, 256, 256, 1) and data type float64